In [2]:
from __future__ import annotations

import json
import os
from collections import defaultdict
from pathlib import Path

import psycopg
from dotenv import load_dotenv
from psycopg.rows import dict_row

ROOT = Path.cwd()
load_dotenv(ROOT / ".env", override=True)

SEARCH_PATH = ["public"]
OUT_PATH = ROOT / "db_schema.json"


def env(*names: str, default: str | None = None) -> str | None:
    for name in names:
        value = os.getenv(name)
        if value:
            return value
    return default


def connect() -> psycopg.Connection:
    return psycopg.connect(
        host=env("DB_HOST_stage", "AUDIT_DB_HOST"),
        port=int(env("AUDIT_DB_PORT_stage", "AUDIT_DB_PORT", default="5432")),
        dbname=env("AUDIT_DB_DATABASE_stage", "AUDIT_DB_DATABASE"),
        user=env("AUDIT_DB_USERNAME_stage", "AUDIT_DB_USERNAME"),
        password=env("AUDIT_DB_PASSWORD_stage", "AUDIT_DB_PASSWORD"),
        row_factory=dict_row,
        connect_timeout=15,
    )


def fetch_all(cur: psycopg.Cursor, sql: str, params=None) -> list[dict]:
    cur.execute(sql, params)
    return list(cur.fetchall())


with connect() as conn:
    conn.execute("SET default_transaction_read_only = on")
    with conn.cursor() as cur:
        dbname = fetch_all(cur, "SELECT current_database() AS name")[0]["name"]
        schemas = fetch_all(
            cur,
            """
            SELECT schema_name
            FROM information_schema.schemata
            WHERE schema_name = ANY(%s)
            ORDER BY schema_name
            """,
            (SEARCH_PATH,),
        )
        schema_names = [row["schema_name"] for row in schemas]

        tables = fetch_all(
            cur,
            """
            SELECT
                n.nspname AS schema_name,
                c.relname AS table_name,
                CASE c.relkind
                    WHEN 'r' THEN 'table'
                    WHEN 'p' THEN 'partitioned_table'
                    WHEN 'v' THEN 'view'
                    WHEN 'm' THEN 'materialized_view'
                    WHEN 'f' THEN 'foreign_table'
                    ELSE c.relkind::text
                END AS table_type,
                obj_description(c.oid, 'pg_class') AS comment
            FROM pg_class c
            JOIN pg_namespace n ON n.oid = c.relnamespace
            WHERE n.nspname = ANY(%s)
              AND c.relkind IN ('r', 'p', 'v', 'm', 'f')
            ORDER BY n.nspname, c.relname
            """,
            (schema_names,),
        )

        columns = fetch_all(
            cur,
            """
            SELECT
                n.nspname AS schema_name,
                c.relname AS table_name,
                a.attname AS column_name,
                a.attnum AS ordinal_position,
                pg_catalog.format_type(a.atttypid, a.atttypmod) AS data_type,
                NOT a.attnotnull AS is_nullable,
                pg_get_expr(ad.adbin, ad.adrelid) AS column_default,
                col_description(c.oid, a.attnum) AS comment,
                a.attidentity AS identity,
                a.attgenerated AS generated
            FROM pg_attribute a
            JOIN pg_class c ON c.oid = a.attrelid
            JOIN pg_namespace n ON n.oid = c.relnamespace
            LEFT JOIN pg_attrdef ad
              ON ad.adrelid = a.attrelid AND ad.adnum = a.attnum
            WHERE n.nspname = ANY(%s)
              AND c.relkind IN ('r', 'p', 'v', 'm', 'f')
              AND a.attnum > 0
              AND NOT a.attisdropped
            ORDER BY n.nspname, c.relname, a.attnum
            """,
            (schema_names,),
        )

        constraints = fetch_all(
            cur,
            """
            SELECT
                n.nspname AS schema_name,
                t.relname AS table_name,
                con.conname AS constraint_name,
                con.contype AS constraint_type,
                pg_get_constraintdef(con.oid) AS definition,
                ARRAY(
                    SELECT att.attname
                    FROM unnest(con.conkey) WITH ORDINALITY AS cols(attnum, ord)
                    JOIN pg_attribute att
                      ON att.attrelid = con.conrelid AND att.attnum = cols.attnum
                    ORDER BY cols.ord
                ) AS columns,
                fn.nspname AS referenced_schema,
                ft.relname AS referenced_table,
                ARRAY(
                    SELECT att.attname
                    FROM unnest(con.confkey) WITH ORDINALITY AS cols(attnum, ord)
                    JOIN pg_attribute att
                      ON att.attrelid = con.confrelid AND att.attnum = cols.attnum
                    ORDER BY cols.ord
                ) AS referenced_columns
            FROM pg_constraint con
            JOIN pg_class t ON t.oid = con.conrelid
            JOIN pg_namespace n ON n.oid = t.relnamespace
            LEFT JOIN pg_class ft ON ft.oid = con.confrelid
            LEFT JOIN pg_namespace fn ON fn.oid = ft.relnamespace
            WHERE n.nspname = ANY(%s)
              AND con.contype IN ('p', 'u', 'f', 'c')
            ORDER BY n.nspname, t.relname, con.conname
            """,
            (schema_names,),
        )

        indexes = fetch_all(
            cur,
            """
            SELECT
                n.nspname AS schema_name,
                t.relname AS table_name,
                i.relname AS index_name,
                ix.indisunique AS is_unique,
                ix.indisprimary AS is_primary,
                am.amname AS access_method,
                pg_get_indexdef(i.oid) AS definition
            FROM pg_index ix
            JOIN pg_class t ON t.oid = ix.indrelid
            JOIN pg_class i ON i.oid = ix.indexrelid
            JOIN pg_namespace n ON n.oid = t.relnamespace
            JOIN pg_am am ON am.oid = i.relam
            WHERE n.nspname = ANY(%s)
            ORDER BY n.nspname, t.relname, i.relname
            """,
            (schema_names,),
        )

schema: dict = {
    "database": dbname,
    "host": env("DB_HOST_stage", "DB_HOST"),
    "search_path": SEARCH_PATH,
    "schemas": {},
}

tables_by_schema: dict[str, dict] = defaultdict(dict)
for table in tables:
    tables_by_schema[table["schema_name"]][table["table_name"]] = {
        "type": table["table_type"],
        "comment": table["comment"],
        "columns": [],
        "primary_key": [],
        "unique_constraints": [],
        "foreign_keys": [],
        "check_constraints": [],
        "indexes": [],
    }

for col in columns:
    table = tables_by_schema.get(col["schema_name"], {}).get(col["table_name"])
    if table is None:
        continue
    identity = col["identity"] or ""
    generated = col["generated"] or ""
    table["columns"].append(
        {
            "name": col["column_name"],
            "position": col["ordinal_position"],
            "data_type": col["data_type"],
            "nullable": col["is_nullable"],
            "default": col["column_default"],
            "identity": identity or None,
            "generated": generated or None,
            "comment": col["comment"],
        }
    )

CONSTRAINT_KIND = {
    "p": "primary_key",
    "u": "unique_constraints",
    "f": "foreign_keys",
    "c": "check_constraints",
}

for con in constraints:
    table = tables_by_schema.get(con["schema_name"], {}).get(con["table_name"])
    if table is None:
        continue
    kind = CONSTRAINT_KIND[con["constraint_type"]]
    payload = {
        "name": con["constraint_name"],
        "columns": list(con["columns"] or []),
        "definition": con["definition"],
    }
    if kind == "primary_key":
        table["primary_key"] = payload["columns"]
        continue
    if kind == "foreign_keys":
        payload["referenced_schema"] = con["referenced_schema"]
        payload["referenced_table"] = con["referenced_table"]
        payload["referenced_columns"] = list(con["referenced_columns"] or [])
    table[kind].append(payload)

for idx in indexes:
    table = tables_by_schema.get(idx["schema_name"], {}).get(idx["table_name"])
    if table is None:
        continue
    table["indexes"].append(
        {
            "name": idx["index_name"],
            "unique": idx["is_unique"],
            "primary": idx["is_primary"],
            "access_method": idx["access_method"],
            "definition": idx["definition"],
        }
    )

for schema_name in schema_names:
    schema["schemas"][schema_name] = {
        "tables": dict(sorted(tables_by_schema[schema_name].items()))
    }

OUT_PATH.write_text(json.dumps(schema, indent=2, default=str) + "\n")

table_count = sum(len(s["tables"]) for s in schema["schemas"].values())
print(f"Wrote {table_count} tables across {len(schema['schemas'])} schemas to {OUT_PATH}")
for schema_name, payload in schema["schemas"].items():
    print(f"  {schema_name}: {len(payload['tables'])} tables")


Wrote 185 tables across 1 schemas to /home/faraz/work/Work/Propqa Chatbot/Chatbot/db_schema.json
  public: 185 tables
